---

Exploring Classification Models pre-processing, training, and validation. Review performance

* Predict sex of the possum
* Utilize Precision/Recall Curves, Area under the curve and confusion matrix
* Look into Unsupervised learning at the end

---

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
from pathlib import Path

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix,roc_auc_score,classification_report,accuracy_score, ConfusionMatrixDisplay

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
possum_dataset = pd.read_csv(Path('data','possum.csv'))
possum_dataset

In [ ]:
possum_dataset.describe()

In [ ]:
nan_counts_per_column = possum_dataset.isna().sum()
print(nan_counts_per_column)

In [ ]:
nan_rows = possum_dataset[possum_dataset.isna().any(axis=1)]
print(nan_rows)

In [ ]:
possum_dataset_processed = possum_dataset.dropna()
possum_dataset_processed

In [ ]:
# extra code – the next 5 lines define the default font sizes
plt.rc('font', size=14)
plt.rc('axes', labelsize=14, titlesize=14)
plt.rc('legend', fontsize=14)
plt.rc('xtick', labelsize=10)
plt.rc('ytick', labelsize=10)

possum_dataset.hist(bins=8, figsize=(12, 8))

In [ ]:
# Generate the scatter matrix
pd.plotting.scatter_matrix(possum_dataset_processed, figsize=(12, 12))
plt.show()

In [ ]:
numerical_columns_list = list(possum_dataset.select_dtypes(include=np.number).columns)
numerical_columns_list

---

* Simple Linear regression model setup above

* Next setup a pipeline

    - Utilize imputer
    - Process the data by scaling using minmax
    - Setup train and test data using age histogram
    - Use different models mentioned in the end to end machine learning project of the Hands-on ML book
    - Select the best model

---

In [ ]:
possum_dataset.sample(n=10)

In [ ]:
possum_dataset['log_age']=np.log(possum_dataset['age'])
possum_dataset['exp_dec_age']=np.exp(-possum_dataset['age'])
possum_dataset['tanh_age']=np.tanh(possum_dataset['age'])

In [ ]:
numerical_columns_list = list(possum_dataset.select_dtypes(include=np.number).columns)
numerical_columns_list

In [ ]:
from sklearn.impute import SimpleImputer

most_frequent_imputer = SimpleImputer(strategy="most_frequent")

In [ ]:
most_frequent_imputer.fit(possum_dataset[numerical_columns_list])

In [ ]:
#help(SimpleImputer)

In [ ]:
most_frequent_imputer.statistics_

In [ ]:
median_imputer = SimpleImputer(strategy="median")

In [ ]:
median_imputer.fit(possum_dataset[numerical_columns_list])
print(median_imputer.statistics_)

In [ ]:
median_imputer

In [ ]:
mean_imputer = SimpleImputer(strategy="mean")

In [ ]:
mean_imputer.fit(possum_dataset[numerical_columns_list])
print(mean_imputer.statistics_)

In [ ]:
plt.figure(figsize=(12, 8))
plt.scatter(numerical_columns_list, 
            most_frequent_imputer.statistics_, 
            label="Most Frequent",
            alpha=0.4)
plt.scatter(numerical_columns_list, 
            median_imputer.statistics_, 
            label="Median",
            alpha=0.4)
plt.scatter(numerical_columns_list, 
            mean_imputer.statistics_, 
            label="Mean",
            alpha=0.4)
plt.legend(loc='upper right', framealpha=0.25)
plt.xlabel("Column Names")
plt.ylabel("Values")
plt.title("Scatter Plot of Different Imputers")
plt.show()

In [ ]:
X = median_imputer.transform(possum_dataset[numerical_columns_list])
print(median_imputer.feature_names_in_)
print(X[:5,:])

In [ ]:
possum_dataset_median_transformed = pd.DataFrame(X, 
                                                 columns=numerical_columns_list,
                                                 index=possum_dataset.index
                                                )
possum_dataset_median_transformed.sample(10)

In [ ]:
# check if imputer worked
nan_counts_per_column_tr = possum_dataset_median_transformed.isna().sum()
print(nan_counts_per_column_tr)

---

* Look into shuffling dataset into test and train utilizing age of possum

---

In [ ]:
possum_dataset_median_transformed.age.hist()

In [ ]:
possum_dataset_median_transformed["age_cat"] = pd.cut(possum_dataset_median_transformed["age"],
                               bins=[0., 2.5, 3.5, 4., 6., np.inf],
                               labels=[1, 2, 3, 4, 5])

In [ ]:
possum_dataset_median_transformed["age_cat"].value_counts().sort_index().plot.bar(rot=0, grid=True)
plt.xlabel("Age category")
plt.ylabel("Number of samples")
plt.show()

In [ ]:
from sklearn.model_selection import StratifiedShuffleSplit

splitter = StratifiedShuffleSplit(n_splits=10, test_size=0.2, random_state=42)
possum_data_strat_splits = []
for train_index, test_index in splitter.split(possum_dataset_median_transformed, 
                                              possum_dataset_median_transformed["age_cat"]):
    possum_data_train_set_n = possum_dataset_median_transformed.iloc[train_index]
    possum_data_test_set_n = possum_dataset_median_transformed.iloc[test_index]
    possum_data_strat_splits.append([possum_data_train_set_n, possum_data_test_set_n])

In [ ]:
possum_data_train_set, possum_data_test_set = possum_data_strat_splits[0]

In [ ]:
possum_data_train_set, possum_data_test_set = train_test_split(
    possum_dataset_median_transformed, 
    test_size=0.2, 
    stratify=possum_dataset_median_transformed["age_cat"], 
    random_state=42)

In [ ]:
possum_data_test_set["age_cat"].value_counts()/len(possum_data_test_set)

In [ ]:
possum_data_train_set["age_cat"].value_counts()/len(possum_data_train_set)

In [ ]:
possum_data_train_set.index

In [ ]:
pd.Series(possum_data_train_set["age_cat"].value_counts()/len(possum_data_train_set)).plot.bar()

In [ ]:
pd.Series(possum_data_test_set["age_cat"].value_counts()/len(possum_data_test_set)).plot.bar()

In [ ]:
for set_ in (possum_data_train_set, possum_data_test_set):
    set_.drop("age_cat", axis=1, inplace=True)

---
* Need to add pre-processing of the dataset
---

In [ ]:
imputed_scaled_possum_train_dataset=pd.DataFrame()
imputed_scaled_possum_train_dataset[numerical_columns_list] = StandardScaler().fit_transform(possum_data_train_set[numerical_columns_list])
imputed_scaled_possum_train_dataset

In [ ]:
imputed_scaled_possum_test_dataset=pd.DataFrame()
imputed_scaled_possum_test_dataset[numerical_columns_list] = StandardScaler().fit_transform(possum_data_test_set[numerical_columns_list])
imputed_scaled_possum_test_dataset

In [ ]:
possum_data_train_set.columns

In [ ]:
selected_columns = list(possum_data_train_set.columns.values)[2:]
selected_columns

In [ ]:
possum_data_train_set[selected_columns].corr().style.background_gradient(cmap='RdBu', vmin=-1, vmax=1).format(precision=3)

---
* Indexed are maintained in the dataframe so gender is carried over.
---

In [ ]:
possum_data_train_set['sex'] = possum_dataset['sex']
possum_data_train_set

In [ ]:
possum_data_test_set['sex'] = possum_dataset['sex']
possum_data_test_set['sex'].values

In [ ]:
possum_data_train_set_encoded = pd.get_dummies(possum_data_train_set, 
                                              columns=['sex'])
print(f"One-Hot Encoded Data using Pandas:\n{possum_data_train_set_encoded}\n")

In [ ]:
possum_data_test_set_encoded = pd.get_dummies(possum_data_test_set, 
                                              columns=['sex'])
print(f"One-Hot Encoded Data using Pandas:\n{possum_data_test_set_encoded}\n")

In [ ]:
possum_data_train_set.columns

In [ ]:
selected_classification_columns=['age', 'hdlngth', 'skullw', 'totlngth', 'taill',
       'footlgth', 'earconch', 'eye', 'chest', 'belly']

---
#### Logistic Regression
---

In [ ]:
from sklearn.linear_model import LogisticRegression

In [ ]:
#help(LogisticRegression)

In [ ]:
logistic_model = LogisticRegression(max_iter=1000, random_state=42)
logistic_model.fit(possum_data_train_set_encoded[selected_classification_columns],
                   possum_data_train_set_encoded['sex_f'])

In [ ]:
logistic_train_pred = logistic_model.predict(possum_data_train_set_encoded[selected_classification_columns])
print("-------------------------------------------------------------------------")
print(f"The accuraccy score:  {np.around(accuracy_score(possum_data_train_set_encoded['sex_f'],
                                                          logistic_train_pred), 4)}")
print("-------------------------------------------------------------------------")
print(f"The Confusion Matrix: \n{confusion_matrix(possum_data_train_set_encoded['sex_f'],
                                                              logistic_train_pred)}")
print("-------------------------------------------------------------------------")
print(f"The Classification Report: \n{classification_report(possum_data_train_set_encoded['sex_f'],
                                                                    logistic_train_pred)}")

In [ ]:
logistic_train_pred = logistic_model.predict(possum_data_train_set_encoded[selected_classification_columns])
print("-------------------------------------------------------------------------")
print(f"The accuraccy score:  {np.around(accuracy_score(possum_data_train_set_encoded['sex_f'],
                                                          logistic_train_pred), 4)}")
print("-------------------------------------------------------------------------")
print(f"The Confusion Matrix: \n{confusion_matrix(possum_data_train_set_encoded['sex_f'],
                                                              logistic_train_pred)}")
print("-------------------------------------------------------------------------")
print(f"The Classification Report: \n{classification_report(possum_data_train_set_encoded['sex_f'],
                                                                    logistic_train_pred)}")

In [ ]:
logistic_test_pred = logistic_model.predict(possum_data_test_set_encoded[selected_classification_columns])
print("-------------------------------------------------------------------------")
print(f"The accuraccy score:  {np.around(accuracy_score(possum_data_test_set_encoded['sex_f'],
                                                          logistic_test_pred), 4)}")
print("-------------------------------------------------------------------------")
print(f"The Confusion Matrix: \n{confusion_matrix(possum_data_test_set_encoded['sex_f'],
                                                              logistic_test_pred)}")
print("-------------------------------------------------------------------------")
print(f"The Classification Report: \n{classification_report(possum_data_test_set_encoded['sex_f'],
                                                                    logistic_test_pred)}")

In [ ]:
logistic_model

---
#### SVM
---

In [ ]:
from sklearn.svm import SVC

In [ ]:
svm_classifier = SVC()
svm_classifier.fit(possum_data_train_set_encoded[selected_classification_columns],
                   possum_data_train_set_encoded['sex_f'])

In [ ]:
svm_train_pred = svm_classifier.predict(possum_data_train_set_encoded[selected_classification_columns])
print("-------------------------------------------------------------------------")
print(f"The accuraccy score:  {np.around(accuracy_score(possum_data_train_set_encoded['sex_f'],
                                                          svm_train_pred), 4)}")
print("-------------------------------------------------------------------------")
print(f"The Confusion Matrix: \n{confusion_matrix(possum_data_train_set_encoded['sex_f'],
                                                              svm_train_pred)}")
print("-------------------------------------------------------------------------")
print(f"The Classification Report: \n{classification_report(possum_data_train_set_encoded['sex_f'],
                                                                    svm_train_pred)}")

In [ ]:
svm_test_pred = svm_classifier.predict(possum_data_test_set_encoded[selected_classification_columns])
print("-------------------------------------------------------------------------")
print(f"The accuraccy score:  {np.around(accuracy_score(possum_data_test_set_encoded['sex_f'],
                                                          svm_test_pred), 4)}")
print("-------------------------------------------------------------------------")
print(f"The Confusion Matrix: \n{confusion_matrix(possum_data_test_set_encoded['sex_f'],
                                                              svm_test_pred)}")
print("-------------------------------------------------------------------------")
print(f"The Classification Report: \n{classification_report(possum_data_test_set_encoded['sex_f'],
                                                                    svm_test_pred)}")

---
#### KNN Classifier
---

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

In [ ]:
knn_classifier = KNeighborsClassifier()
knn_classifier.fit(possum_data_test_set_encoded[selected_classification_columns],
                   possum_data_test_set_encoded['sex_f'])

In [ ]:
knn_classifier_test_pred = knn_classifier.predict(possum_data_test_set_encoded[selected_classification_columns])
print("-------------------------------------------------------------------------")
print(f"The accuraccy score:  {np.around(accuracy_score(possum_data_test_set_encoded['sex_f'],
                                                          knn_classifier_train_pred), 4)}")
print("-------------------------------------------------------------------------")
print(f"The Confusion Matrix: \n{confusion_matrix(possum_data_train_set_encoded['sex_f'],
                                                              knn_classifier_train_pred)}")
print("-------------------------------------------------------------------------")
print(f"The Classification Report: \n{classification_report(possum_data_train_set_encoded['sex_f'],
                                                                    knn_classifier_train_pred)}")

In [ ]:
knn_classifier_test_pred = knn_classifier.predict(possum_data_test_set_encoded[selected_classification_columns])
print("-------------------------------------------------------------------------")
print(f"The accuraccy score:  {np.around(accuracy_score(possum_data_test_set_encoded['sex_f'],
                                                          knn_classifier_test_pred), 4)}")
print("-------------------------------------------------------------------------")
print(f"The Confusion Matrix: \n{confusion_matrix(possum_data_test_set_encoded['sex_f'],
                                                              knn_classifier_test_pred)}")
print("-------------------------------------------------------------------------")
print(f"The Classification Report: \n{classification_report(possum_data_test_set_encoded['sex_f'],
                                                                    knn_classifier_test_pred)}")

---
#### Titanic dataset exploration from the book Hands on ML with Scikit-Learn by A Geron

* Generating metrics myself and other models from the solution
---

In [ ]:
import tarfile
import urllib.request

In [ ]:
def load_titanic_data():
    tarball_path = Path("datasets/titanic.tgz")
    if not tarball_path.is_file():
        Path("datasets").mkdir(parents=True, exist_ok=True)
        url = "https://github.com/ageron/data/raw/main/titanic.tgz"
        urllib.request.urlretrieve(url, tarball_path)
        with tarfile.open(tarball_path) as titanic_tarball:
            titanic_tarball.extractall(path="datasets")
    return [pd.read_csv(Path("datasets/titanic") / filename)
            for filename in ("train.csv", "test.csv")]

In [ ]:
train_data, test_data = load_titanic_data()

In [ ]:
train_data.head(10)

In [ ]:
train_data = train_data.set_index("PassengerId")
test_data = test_data.set_index("PassengerId")

In [ ]:
test_data.describe()

In [ ]:
train_data.describe()

In [ ]:
train_data.info()

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

num_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

In [ ]:
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder

In [ ]:
cat_pipeline = Pipeline([
        ("ordinal_encoder", OrdinalEncoder()),    
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("cat_encoder", OneHotEncoder(sparse_output=False)),
    ])

In [ ]:
from sklearn.compose import ColumnTransformer

num_attribs = ["Age", "SibSp", "Parch", "Fare"]
cat_attribs = ["Pclass", "Sex", "Embarked"]

preprocess_pipeline = ColumnTransformer([
        ("num", num_pipeline, num_attribs),
        ("cat", cat_pipeline, cat_attribs),
    ])

In [ ]:
X_train = preprocess_pipeline.fit_transform(train_data)
X_train

In [ ]:
y_train = train_data["Survived"]

In [ ]:
from sklearn.ensemble import RandomForestClassifier

In [ ]:
forest_clf = RandomForestClassifier(n_estimators=100, random_state=42)
forest_clf.fit(X_train, y_train)

In [ ]:
X_test = preprocess_pipeline.transform(test_data)
y_pred = forest_clf.predict(X_test)

In [ ]:
test_data.columns

In [ ]:
from sklearn.model_selection import cross_val_score

In [ ]:
forest_scores = cross_val_score(forest_clf, X_train, y_train, cv=10)
round(forest_scores.mean(),4)

In [ ]:
svm_clf = SVC(gamma="auto")
svm_scores = cross_val_score(svm_clf, X_train, y_train, cv=10)
round(svm_scores.mean(),4)

In [ ]:
logistic_scores = cross_val_score(logistic_model, X_train, y_train, cv=10)
round(logistic_scores.mean(),4)

In [ ]:
knn_scores = cross_val_score(knn_classifier, X_train, y_train, cv=10)
round(knn_scores.mean(),4)

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
gb_model=GradientBoostingClassifier(n_estimators=10, 
                                    random_state=42)
gb_scores = cross_val_score(gb_model, X_train, y_train, cv=10)
round(gb_scores.mean(),4)

In [ ]:
from sklearn.tree import DecisionTreeClassifier
dt_model=DecisionTreeClassifier(max_depth=4, 
                                    random_state=42)
dt_scores = cross_val_score(dt_model, X_train, y_train, cv=10)
round(dt_scores.mean(),4)

In [ ]:
cross_val_df = pd.DataFrame({
    "Random Forest": pd.Series(forest_scores),
    "Gradient Boosted": pd.Series(gb_scores),
    "SVM": pd.Series(svm_scores),
    "Logistic": pd.Series(logistic_scores),
    "KNN Classifier": pd.Series(knn_scores),
    "Decision Tree": pd.Series(dt_scores)
    })

In [ ]:
cross_val_df

In [ ]:
plt.figure(figsize=(12, 8))
sns.swarmplot(data = cross_val_df)
plt.show()

In [ ]:
plt.figure(figsize=(12, 8))
sns.boxplot(data = cross_val_df)
plt.show()

In [ ]:
plt.figure(figsize=(12, 8))
sns.violinplot(data = cross_val_df)
plt.show()

* Focus on Random Forest, Gradient Boosted and SVM
     - doing grid search for best hyperparameters

In [ ]:
from sklearn.model_selection import GridSearchCV

In [ ]:
titanic_dataset_forest_pipeline = Pipeline([("rf_clf", RandomForestClassifier(random_state=42))
])


In [ ]:
param_grid_forest = [{'rf_clf__n_estimators': [20, 40, 60, 80, 100, 120],
                     'rf_clf__max_depth': [None, 1, 2, 3, 4],
                     'rf_clf__min_samples_leaf': [1, 2, 4, 8, 16, 32, 64]}]
grid_search_forest = GridSearchCV(titanic_dataset_forest_pipeline, 
                              param_grid_forest, 
                              cv=10,
                              scoring='accuracy')
grid_search_forest.fit(X_train, y_train)

In [ ]:
grid_search_forest.best_params_

In [ ]:
final_model_forest = grid_search_forest.best_estimator_ 
feature_importances = final_model_forest["rf_clf"].feature_importances_
feature_importances.round(2)

In [ ]:
train_data.columns

In [ ]:
titanic_dataset_gb_pipeline = Pipeline([("gb_clf", GradientBoostingClassifier(random_state=42))
])


In [ ]:
param_grid_gb = [{'gb_clf__n_estimators': [5, 10, 15, 20, 25],
                  'gb_clf__learning_rate': [0.05, 0.1, 0.2],
                     'gb_clf__max_depth': [None, 1, 2, 3, 4],
                     'gb_clf__min_samples_leaf': [1, 2, 4, 8, 16, 32]}]
grid_search_gb = GridSearchCV(titanic_dataset_gb_pipeline, 
                              param_grid_gb, 
                              cv=10,
                              scoring='accuracy')
grid_search_gb.fit(X_train, y_train)

In [ ]:
grid_search_gb.best_params_

In [ ]:
final_model_gb = grid_search_gb.best_estimator_ 
feature_importances = final_model_gb["gb_clf"].feature_importances_
feature_importances.round(2)

In [ ]:
titanic_dataset_svm_pipeline = Pipeline([("svm_clf", SVC(gamma="auto", random_state=42))
])

In [ ]:
param_grid_svm = [{'svm_clf__C': [1.0, 2.0, 4.0, 8.0, 16.0],
                  'svm_clf__degree': [1, 2, 3],
                     #'svm_clf__class_weight': [None, 'balanced'],
                     'svm_clf__kernel': ['linear', 'poly', 'rbf']}]
grid_search_svm = GridSearchCV(titanic_dataset_svm_pipeline, 
                              param_grid_svm, 
                              cv=10,
                              scoring='accuracy')
grid_search_svm.fit(X_train, y_train)

In [ ]:
grid_search_svm.best_params_

In [ ]:
titanic_dataset_dt_pipeline = Pipeline([("dt_clf", DecisionTreeClassifier(random_state=42))
])

In [ ]:
param_grid_dt = [{'dt_clf__max_depth': [1, 2, 3, 4, 5, 6],
                  'dt_clf__min_samples_split': [2, 4, 8, 16, 32, 64, 128],
                     'dt_clf__max_features': [None, 1, 2, 3, 4, 5]}]
grid_search_dt = GridSearchCV(titanic_dataset_dt_pipeline, 
                              param_grid_dt, 
                              cv=10,
                              scoring='accuracy')
grid_search_dt.fit(X_train, y_train)

In [ ]:
grid_search_dt.best_params_

In [ ]:
forest_clf = RandomForestClassifier(min_samples_leaf=4,
                                    n_estimators=100, 
                                    random_state=42)
forest_clf.fit(X_train, y_train)
forest_scores = cross_val_score(forest_clf, X_train, y_train, cv=10)
round(forest_scores.mean(),4)

In [ ]:
gb_model=GradientBoostingClassifier(learning_rate=0.2,
                                    max_depth=4,
                                    min_samples_leaf=2,
                                    n_estimators=25,
                                    random_state=42)
gb_model.fit(X_train, y_train)
gb_scores = cross_val_score(gb_model, X_train, y_train, cv=10)
round(gb_scores.mean(),4)

In [ ]:
svm_clf = SVC(C=2.0, degree=3, kernel='poly', gamma="auto", random_state=42)
svm_clf.fit(X_train, y_train)
svm_scores = cross_val_score(svm_clf, X_train, y_train, cv=10)
round(svm_scores.mean(),4)

In [ ]:
dt_clf = DecisionTreeClassifier(max_depth=5,
                                max_features=5, 
                                min_samples_split=2,
                                random_state=42)
dt_clf.fit(X_train, y_train)
dt_scores = cross_val_score(dt_clf, X_train, y_train, cv=10)
round(dt_scores.mean(),4)

* 2% to 4% improvement using grid search
* Large improvement for Decision Tree from 0.80 to 0.83
* Since a white box model on par with black box models better to go with decision tree
     - Better tunability or interpretability
     - Understand the inner workings or decisions made by the model
     - Decision Tree also has tighter distribution for 10-fold cross validation

In [ ]:
cross_val_df = pd.DataFrame({
    "Random Forest": pd.Series(forest_scores),
    "Gradient Boosted": pd.Series(gb_scores),
    "SVM": pd.Series(svm_scores),
    "Decision Tree": pd.Series(dt_scores),
    })

In [ ]:
cross_val_df

In [ ]:
plt.figure(figsize=(12, 8))
sns.swarmplot(data = cross_val_df)
plt.show()

In [ ]:
plt.figure(figsize=(12, 8))
sns.boxplot(data = cross_val_df)
plt.show()

In [ ]:
plt.figure(figsize=(12, 8))
sns.violinplot(data = cross_val_df)
plt.show()

In [ ]:
forest_train_pred = forest_clf.predict(X_train)
print("-------------------------------------------------------------------------")
print(f"The accuraccy score:  {np.around(accuracy_score(y_train,
                                                          forest_train_pred), 4)}")
print("-------------------------------------------------------------------------")
print(f"The Confusion Matrix: \n{confusion_matrix(y_train,
                                                              forest_train_pred)}")
print("-------------------------------------------------------------------------")
print(f"The Classification Report: \n{classification_report(y_train,
                                                                    forest_train_pred)}")

In [ ]:
# Create a Pandas DataFrame
df_cm = pd.DataFrame(confusion_matrix(y_train, forest_train_pred), 
                     index=np.unique(y_train), 
                     columns=np.unique(y_train))

# Plot the heatmap
plt.figure(figsize=(8, 6))
sns.heatmap(df_cm, annot=True, fmt="d", cmap="Blues", linewidths=.5)
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix")
plt.show()

In [ ]:
gb_train_pred = gb_model.predict(X_train)
print("-------------------------------------------------------------------------")
print(f"The accuraccy score:  {np.around(accuracy_score(y_train,
                                                          gb_train_pred), 4)}")
print("-------------------------------------------------------------------------")
print(f"The Confusion Matrix: \n{confusion_matrix(y_train,
                                                              gb_train_pred)}")
print("-------------------------------------------------------------------------")
print(f"The Classification Report: \n{classification_report(y_train,
                                                                    gb_train_pred)}")

In [ ]:
# Create a Pandas DataFrame
df_cm = pd.DataFrame(confusion_matrix(y_train, gb_train_pred), 
                     index=np.unique(y_train), 
                     columns=np.unique(y_train))

# Plot the heatmap
plt.figure(figsize=(8, 6))
sns.heatmap(df_cm, annot=True, fmt="d", cmap="Blues", linewidths=.5)
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix")
plt.show()

In [ ]:
svm_train_pred = svm_clf.predict(X_train)
print("-------------------------------------------------------------------------")
print(f"The accuraccy score:  {np.around(accuracy_score(y_train,
                                                          svm_train_pred), 4)}")
print("-------------------------------------------------------------------------")
print(f"The Confusion Matrix: \n{confusion_matrix(y_train,
                                                              svm_train_pred)}")
print("-------------------------------------------------------------------------")
print(f"The Classification Report: \n{classification_report(y_train,
                                                                    svm_train_pred)}")

In [ ]:
# Create a Pandas DataFrame
df_cm = pd.DataFrame(confusion_matrix(y_train, svm_train_pred), 
                     index=np.unique(y_train), 
                     columns=np.unique(y_train))

# Plot the heatmap
plt.figure(figsize=(8, 6))
sns.heatmap(df_cm, annot=True, fmt="d", cmap="Blues", linewidths=.5)
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix")
plt.show()

In [ ]:
dt_train_pred = dt_clf.predict(X_train)
print("-------------------------------------------------------------------------")
print(f"The accuraccy score:  {np.around(accuracy_score(y_train,
                                                          dt_train_pred), 4)}")
print("-------------------------------------------------------------------------")
print(f"The Confusion Matrix: \n{confusion_matrix(y_train,
                                                              dt_train_pred)}")
print("-------------------------------------------------------------------------")
print(f"The Classification Report: \n{classification_report(y_train,
                                                                    dt_train_pred)}")

In [ ]:
# Create a Pandas DataFrame
df_cm = pd.DataFrame(confusion_matrix(y_train, dt_train_pred), 
                     index=np.unique(y_train), 
                     columns=np.unique(y_train))

# Plot the heatmap
plt.figure(figsize=(8, 6))
sns.heatmap(df_cm, annot=True, fmt="d", cmap="Blues", linewidths=.5)
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix")
plt.show()